# Risk analysis and segmentation

## General risk profile

In [ ]:
n_loans = len(df_clean)
status_counts = df_clean["loan_status"].value_counts().sort_index()
default_rate = df_clean["loan_status"].mean()

print(f"Total number of loans: {n_loans:,}")
print(f"Non-default loans: {status_counts.get(0, 0):,}")
print(f"Default loans: {status_counts.get(1, 0):,}")
print(f"Default rate: {default_rate:.2%}")

Out of all the records, there are 6,939 observations that are default loans taking up 21.91% of the cleaned dataset, suggesting a moderate imbalance between default and non-default representations. What if this imbalance is also present in different categories, suggesting that some factors are associated with higher risk than others?

The next part of the notebook will be investigating, if there are categories that contribute to higher default rates in the data.

## Risk analysis

In [ ]:
def risk_by(column):
    return df_clean.groupby(column).agg(
        loan_count=("loan_status", "size"),
        default_count=("loan_status", "sum"),
        default_rate=("loan_status", "mean")
    )

risk_by_grade = risk_by("loan_grade")
display(risk_by_grade)

Since loan grades represent the likelihood of a borrower defaulting, it would be logical to assume that higher default rates will be observed in grades associated with higher risk (letters placed later in the alphabet). The output above seems to confirm this theory as there is a consistent increase in the default rate towards higher risk categories, however, it is noteable that these groups are also less represented in the data, raising a bit of uncertainty about the verification of the assumption.

In [ ]:
for col in ["loan_intent", "person_home_ownership", "cb_person_default_on_file"]:
    risk_table = risk_by(col)
    display(risk_table)

The results show that higher default rates occur more in groups with the loan purpose of debt consolidation, home improvement and settling medical costs. Also, the default rate is significantly higher in groups with rentals or with the "other" home ownership category, however, since there are only 105 records observed in the latter, we cannot be certain that the default rate would not change with more data at hand. While the distibution of records is also skewed within the categories of the cb_person_default_on_file column, there is an observed increase in default rates of people with a history of defaulting than those without.

Now, we will be investigating, if other factors like interest rate or loan burden differentiate the data as well. 

In [ ]:
bins = [0, 0.10, 0.20, 0.30, 0.40, 0.50, 1.00]

df_clean["loan_income_band"] = pd.cut(
    df_clean["loan_percent_income"], bins=bins, right=True, include_lowest=True)

burden_analysis = (df_clean.groupby("loan_income_band", observed=False)
    .agg(
        loan_count=("loan_status", "size"),
        default_count=("loan_status", "sum"),
        default_rate=("loan_status", "mean")
    )
)
display(burden_analysis)